In [1]:
%load_ext cudf.pandas
import dias.rewriter
import numpy as np  # linear algebra
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [2]:
start = time.time()

In [3]:
# load & cleanup
file = "/home/dias-benchmarks/notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/input/indian-startup-recognized-by-dpiit/Startup_Counts_Across_India.csv"
df = pd.read_csv(file)

# -- STEFANOS -- Replicate Data

In [4]:
factor = 3000
df = pd.concat([df] * factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 17673000 entries, 0 to 5890
Data columns (total 5 columns):
 #   Column    Dtype
---  ------    -----
 0   S No.     int64
 1   Year      int64
 2   State     object
 3   Industry  object
 4   Count     int64
dtypes: int64(3), object(2)
memory usage: 1.1+ GB


In [5]:
df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

,Year,State,Industry,Count
0,2022,Andaman and Nicobar Islands,Agriculture,1
1,2022,Andaman and Nicobar Islands,AR VR (Augmented + Virtual Reality),1
2,2022,Andaman and Nicobar Islands,Construction,1
3,2022,Andaman and Nicobar Islands,Internet of Things,1
4,2022,Andaman and Nicobar Islands,Marketing,1


In [6]:
# Industry sub-categories for environmental & AI startups
env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai = ["AI", "Robotics", "Computer Vision"]

# combined df - environmental & AI startups only
df_ea = df.loc[(df["Industry"].isin(env)) | (df["Industry"].isin(ai))].reset_index(
    drop=True, inplace=False
)


# custom function to set Main Industry
def set_MainIndustry(ind):
    if ind in env:
        return "ENV"
    else:
        return "AI"


# adding a new column
df_ea["MainIndustry"] = df_ea.Industry.apply(lambda x: set_MainIndustry(x))

# basic stats
print(
    f"A total of {df_ea.shape[0]} startups were started in India between 2016 & 2022, out of which {df_ea.groupby('MainIndustry').size()['ENV']} are environmental related & {df_ea.groupby('MainIndustry').size()['AI']} are AI startups."
)

A total of 2361000 startups were started in India between 2016 & 2022, out of which 1473000 are environmental related & 888000 are AI startups.


In [7]:
end = time.time()
print(end - start)

4.058693885803223


In [8]:
4.015127658843994, 4.063482999801636, 4.058693885803223

(7.942842721939087, 7.993764877319336, 8.11342191696167)